In [1]:
import sys
import os

# localizar caminho automaticamente
import pandas as pd

from rdkit import RDConfig
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))

import sascorer
from rdkit import Chem

In [2]:
# ===== CONFIGURAÇÕES =====
# Defina aqui os caminhos e parâmetros

# Diretório base
base_dir = '/Users/francisco/Library/CloudStorage/OneDrive-Pessoal/Documentos/LabMol/Doutorado/SWE/RESULTS_LigBuilder/Build_HsDHODH_MAY/result/'

# Arquivo de entrada
input_file = base_dir + 'dados_sem_duplicatas.csv'

# Arquivos de saída
output_file_completo = base_dir + 'dados_com_sa_score.csv'
output_file_filtrado = base_dir + 'dados_sa_score_filtrado.csv'

# Nome da coluna de SMILES
coluna_smiles = 'SMILES'

# Threshold do SA Score (moléculas com SA Score < threshold receberão valor 1)
threshold = 4.0
# =========================

In [3]:
# Carregar o dataset
df = pd.read_csv(input_file)

print(f"Dataset carregado: {len(df)} linhas")
df.head()

Dataset carregado: 3147 linhas


,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical
0,OC(=O)Cc1cc([nH]c1c1ccc(c(c1)C(=O)C)CCC)C(=O)O...,23/000000290,InChI=1S/C18H19NO5/c1-3-4-11-5-6-12(7-14(11)10...,C18H18NO5,328,1.98,5.49,7.32,0.0,-160
1,Oc1ccc(c(c1)C(=O)CO)CC[C@@H](c1nc2c([nH]1)cccc...,28/000000683,InChI=1S/C18H18N2O4/c21-10-17(24)13-9-12(22)7-...,C18H18N2O4,326,1.24,5.08,7.36,0.0,-150
2,OC(=O)Cc1cc([nH]c1c1ccc(cc1O)CCC)C(=O)O\tNo_6_1,23/000000341,InChI=1S/C16H17NO5/c1-2-3-9-4-5-11(13(18)6-9)1...,C16H16NO5,302,1.74,5.08,6.41,0.0,-100
3,Oc1ccc(c(c1)C(=O)CO)CC[C@@H](c1nc2c(o1)cccc2C)...,28/000000698,InChI=1S/C19H19NO5/c1-11-3-2-4-17-18(11)20-19(...,C19H19NO5,341,1.36,5.07,8.86,0.0,-170
4,Oc1ccc(c(c1)C(=O)CO)CC[C@@H](c1nc2c(o1)cc(cc2)...,28/000000709,InChI=1S/C20H21NO5/c1-2-12-3-7-16-19(9-12)26-2...,C20H21NO5,355,1.61,5.02,6.15,0.0,-170


In [4]:
# Função para calcular o SA Score
def calculate_sa_score(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return sascorer.calculateScore(mol)
        else:
            return None
    except:
        return None

# Aplicar a função na coluna de SMILES
df['SA_Score'] = df[coluna_smiles].apply(calculate_sa_score)

# Criar coluna binária baseada no threshold
df['SA_Score_Binary'] = (df['SA_Score'] < threshold).astype(int)

# Visualizar resultado
df.head()

,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical,SA_Score,SA_Score_Binary
0,OC(=O)Cc1cc([nH]c1c1ccc(c(c1)C(=O)C)CCC)C(=O)O...,23/000000290,InChI=1S/C18H19NO5/c1-3-4-11-5-6-12(7-14(11)10...,C18H18NO5,328,1.98,5.49,7.32,0.0,-160,2.603156,1
1,Oc1ccc(c(c1)C(=O)CO)CC[C@@H](c1nc2c([nH]1)cccc...,28/000000683,InChI=1S/C18H18N2O4/c21-10-17(24)13-9-12(22)7-...,C18H18N2O4,326,1.24,5.08,7.36,0.0,-150,2.979825,1
2,OC(=O)Cc1cc([nH]c1c1ccc(cc1O)CCC)C(=O)O\tNo_6_1,23/000000341,InChI=1S/C16H17NO5/c1-2-3-9-4-5-11(13(18)6-9)1...,C16H16NO5,302,1.74,5.08,6.41,0.0,-100,2.606850,1
3,Oc1ccc(c(c1)C(=O)CO)CC[C@@H](c1nc2c(o1)cccc2C)...,28/000000698,InChI=1S/C19H19NO5/c1-11-3-2-4-17-18(11)20-19(...,C19H19NO5,341,1.36,5.07,8.86,0.0,-170,3.170634,1
4,Oc1ccc(c(c1)C(=O)CO)CC[C@@H](c1nc2c(o1)cc(cc2)...,28/000000709,InChI=1S/C20H21NO5/c1-2-12-3-7-16-19(9-12)26-2...,C20H21NO5,355,1.61,5.02,6.15,0.0,-170,3.144484,1


In [5]:
# Contar quantos 1s e 0s na coluna binária
contagem = df['SA_Score_Binary'].value_counts().sort_index()
print(contagem)

SA_Score_Binary
0     164
1    2983
Name: count, dtype: int64


In [6]:
# Salvar dataset completo com SA Score
df.to_csv(output_file_completo, index=False)
print(f"Dataset completo salvo em: {output_file_completo}")

Dataset completo salvo em: /Users/francisco/Library/CloudStorage/OneDrive-Pessoal/Documentos/LabMol/Doutorado/SWE/RESULTS_LigBuilder/Build_HsDHODH_MAY/result/dados_com_sa_score.csv


In [7]:
# Filtrar apenas moléculas com SA_Score_Binary = 1 (SA Score < threshold)
df_filtered = df[df['SA_Score_Binary'] == 1].copy()

print(f"Total de moléculas filtradas: {len(df_filtered)} (SA Score < {threshold})")

# Salvar o dataframe filtrado em CSV
df_filtered.to_csv(output_file_filtrado, index=False)

print(f"Arquivo filtrado salvo em: {output_file_filtrado}")

Total de moléculas filtradas: 2983 (SA Score < 4.0)
Arquivo filtrado salvo em: /Users/francisco/Library/CloudStorage/OneDrive-Pessoal/Documentos/LabMol/Doutorado/SWE/RESULTS_LigBuilder/Build_HsDHODH_MAY/result/dados_sa_score_filtrado.csv
